In [ ]:
using Symbolics, Bessels, Latexify

In [ ]:
@variables a::Complex, b::Complex, ω_d, ω_p, δ_a, δ_b, ξ, r, t

a_st = conj(a)
b_st = conj(b)

n_max = 3
k_max = 3 

g_coeffs, = @variables g[1:n_max, 1:k_max]

rot_a = exp(-1im*(ω_p+ω_d)*t/2)
rot_b = exp(-1im*ω_d*t)

summand_int = a*rot_a + a_st*conj(rot_a) + r*((b+ξ)*rot_b+(b_st+ξ)*conj(rot_b))
summand = [[summand_int^n * 2*cos(k*ω_p*t) for k in 1:k_max] for n in 1:n_max]

H= δ_a*a_st*a + δ_b*b_st*b + sum(g_coeffs[n, k] * summand[n][k] for n in 1:n_max, k in 1:n_max)

In [ ]:
H_array = Matrix{Complex{Num}}(undef, 2*n_max + 1, 2*(2*k_max + n_max) + 1)
middle = (n_max + 1,1+ (2*k_max + n_max))

In [ ]:
H*rot_a

In [ ]:
for n in 1:1
    for k in 1:1
        H_n_k = expand(g_coeffs[n, k] * summand[n][k])
        print(H_n_k)
        println(Symbolics.coeff(H_n_k,cos(t*ω_p)))
    end
end

In [ ]:
for i in -n_max:n_max
    for j in -2k_max-n_max:2k_max+n_max
        H_array[middle[1]+i, middle[2]+j]
    end
end

In [ ]:
# Extract terms not depending on `t`
terms = Symbolics.operand.(H isa Num && Symbolics.is_addition(H) ? H : [H])


In [ ]:
time_indep_terms = filter(term -> !Symbolics.hasvariable(term, t), terms)

# Reconstruct the expression
H_time_indep = sum(time_indep_terms)

In [ ]:
H_exp = expand(H)

In [ ]:
Symbolics.coeff(H_exp,cos(t*ω_p))

In [ ]:
using Symbolics

@variables t a b c

# Define your symbolic expression
H = a + b*sin(t) + c*t^2

# Check if it's a sum of terms
if Symbolics.operation(H) == :+
    terms = Symbolics.arguments(H)
else
    terms = [H]  # not a sum, just a single term
end

# Filter out time-dependent terms
time_indep_terms = filter(term -> !Symbolics.hasvars(term, t), terms)

# Recombine them
H_time_indep = sum(time_indep_terms)


In [ ]:
function coeff_2(p, sym=nothing)
    # if `sym` is a product, iteratively compute the coefficient w.r.t. each term in `sym`
    if iscall(value(sym)) && operation(value(sym)) === (*)
        for t in arguments(value(sym))
            @assert !(t isa Number) "`coeff(p, sym)` does not allow `sym` containing numerical factors"
            p = coeff(p, t)
        end
        return p
    end
            
    p, sym = value(p), value(sym)

    if isequal(sym, 1)
        sym = nothing
    end

    if issym(p) || isterm(p)
        sym === nothing ? 0 : Int(isequal(p, sym))
    elseif ispow(p)
        sym === nothing ? 0 : Int(isequal(p, sym))
    elseif isadd(p)
        if sym===nothing
            p.coeff
        else
            sum(coeff(k, sym) * v for (k, v) in p.dict)
        end
    elseif ismul(p)
        args = arguments(p)
        coeffs = map(a->coeff(a, sym), args)
        if all(iszero, coeffs)
            return 0
        else
            @views prod(Iterators.flatten((coeffs[findall(!iszero, coeffs)], args[findall(iszero, coeffs)])))
        end
    elseif isdiv(p)
        numerator, denominator = arguments(p)
        if !occursin(sym, denominator)
            coeff(numerator, sym) / denominator
        else
            throw(DomainError(p, "coeff on fractions is not yet implemented."))
        end
    else
        p isa Number && return sym === nothing ? p : 0
        p isa Symbolic && return coeff(p, sym)
        throw(DomainError(p, "Datatype $(typeof(p)) not accepted."))
    end
end

In [ ]:
coeff_2(H,b)

In [ ]:
using Symbolics

@variables x y
expr = x^2 + sin(y) + 3

Symbolics.arguments(unwrap(expr))
# Output: Any[x^2, sin(y), 3]
